# motion_scale sweep

sweep motion_scale in {0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0} on 8 prompts, seed fixed.

we look at how motion-score, temporal CLIP drift, and subjective smoothness change.

In [ ]:
import numpy as np, os, imageio, pandas as pd
from src.inference.t2v import run as run_single
from src.eval.clipsim_temporal import CLIPTemporal
from src.eval.motion_score import motion_score

PROMPTS = ['a cat surfing on a rainbow', 'astronaut planting a flag', 'coffee pour slow motion',
           'city rain neon', 'origami crane unfolding', 'dragon over mountain',
           'kids running through sprinklers', 'timelapse city sunrise']
SCALES = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]


In [ ]:
os.makedirs('runs/motion_sweep', exist_ok=True)
for i, p in enumerate(PROMPTS):
    for s in SCALES:
        out = f'runs/motion_sweep/p{i}_s{s}.mp4'
        run_single(prompt=p, out_path=out, motion_scale=s, seed=1729)


In [ ]:
clip = CLIPTemporal()
rows = []
for s in SCALES:
    ms, ct_std = [], []
    for i, p in enumerate(PROMPTS):
        f = np.stack(list(imageio.get_reader(f'runs/motion_sweep/p{i}_s{s}.mp4')))
        ms.append(motion_score(f)['mean'])
        c = clip.score(p, f); ct_std.append(c['std'])
    rows.append({'motion_scale': s, 'motion_mean': np.mean(ms), 'clip_std': np.mean(ct_std)})
pd.DataFrame(rows)
